In [51]:
import torch 
import os
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/cifar10-features")
VARIANT = "mlpl1"

def open_data_inmemory(variant: str, data_path: str, split: str) -> tuple[torch.Tensor]:
    dir = data_path/f"{variant}"
    tensors_x = []
    tensors_y = []
    files = list(os.listdir(dir))
    files_ids = set(int(x.split("_")[-1].split(".")[0]) for x in files if split in x)
    for file_id in files_ids:
        x_name = f"{split}_{variant}_{file_id}.pt"
        y_name = f"{split}_y_{file_id}.pt"
        tensors_x.append(torch.load(dir/x_name))
        tensors_y.append(torch.load(dir/y_name))
    return torch.cat(tensors_x), torch.cat(tensors_y)

def open_rescaled_data(variant: str, data_path: str):
    train_x, train_y = open_data_inmemory(variant, data_path, "train")
    train_mean, train_std = train_x.mean(), train_x.std()
    eval_x, eval_y = open_data_inmemory(variant, data_path, "eval")
    train_x_rescaled = (train_x - train_mean) / train_std
    eval_x_rescaled = (eval_x - train_mean) / train_std
    return (train_x_rescaled, train_y), (eval_x_rescaled, eval_y)

In [52]:
(train_x_rescaled, train_y), (eval_x_rescaled, eval_y) = open_rescaled_data(VARIANT, DATA_DIR)

In [ ]:
import jax
import numpy as np
from pathlib import Path

import jax.numpy as jnp

out_dir = DATA_DIR / VARIANT
out_dir.mkdir(parents=True, exist_ok=True)

def torch_to_jax(t: torch.Tensor):
    return jnp.array(t.detach().cpu().numpy())

train_x_jax = torch_to_jax(train_x_rescaled)
train_y_jax = torch_to_jax(train_y).astype(jnp.int32)
eval_x_jax = torch_to_jax(eval_x_rescaled)
eval_y_jax = torch_to_jax(eval_y).astype(jnp.int32)

np.savez_compressed(out_dir / "train_set.npz",
                    x=jax.device_get(train_x_jax),
                    y=jax.device_get(train_y_jax))
np.savez_compressed(out_dir / "eval_set.npz",
                    x=jax.device_get(eval_x_jax),
                    y=jax.device_get(eval_y_jax))

print("Saved JAX datasets to:", out_dir)



Saved JAX datasets to: ../data/cifar10-features/mlpl1
